In [ ]:
import pandas as pd
from pipeline import IndicatorSpec, build_feature_df
from data import fetch_binance_klines
from simulation.rules import Rule, RuleGroup, ALL, ANY
from simulation.rule_features import add_last_n_compare, add_cross_compare
from simulation.simulator import Strategy, SimConfig, run_simulation, align_timeframes
from plot_simulation import plot_simulation
from plotter import plot_interactive, PlotConfig, IndicatorSpec
from mtf_plot import make_plot_indicators, assert_plot_columns_exist
from features.ema_spreads import EmaSpreadSpec, add_ema_spread_features
from features.cross_through import CrossThroughSpec, add_cross_through_features
from features.stochastic_signals import add_stochastic_signal_features
from features.ema_compression import EMACompressionSpec, add_ema_compression_features, range_slope_strong_for_n_tf_candles
from simulation.rules import Ctx
from analytics import (
    trades_to_frame,
    plot_balance_by_trade,
    plot_trade_pnl_bars,
    trade_summary_table
)
from plot_toggles import (
    PlotToggle,
    make_plot_indicators_from_toggles,
    assert_plot_columns_exist,
)

from ema_diagnostic_plots import (
    ema_pair_spread_panel,
    ema_group_spread_panel,
    ema_cut_through_panel,
)
from simulation.simulator import (
    TradeExitConfig,
    StopLossConfig,
    PositionSizingConfig,
    TakeProfitConfig,
)

In [ ]:
symbol = "BTCUSDT"
market = "spot"
tz = "Asia/Karachi"

# Start earlier for EMA warmup.
fetch_start = "2025-11-01"
sim_start = "2026-05-06"
end = None

In [ ]:
# ============================================================
# STOCHASTIC Configs
# ============================================================

STOCH_CFG = {
    "k_length": 14,
    "d_length": 3,
    "smooth": 3,
}

In [ ]:
### 1 Minute Indicators ###
EMA20_1m = "1m__ema__EMA_20"
EMA50_1m = "1m__ema__EMA_50"
EMA100_1m = "1m__ema__EMA_100"
EMA150_1m = "1m__ema__EMA_150"
EMA200_1m = "1m__ema__EMA_200"
MACD_1M = "1m__macd_8_21_5__MACD"
MACD_SIGNAL_1M = "1m__macd_8_21_5__SIGNAL"
MACD_HIST_1M = "1m__macd_8_21_5__HIST"
BULL_DIV_1m = "1m__rsi14__BULL_DIV"
BEAR_DIV_1m = "1m__rsi14__BEAR_DIV"
BULL_START_RSI_1m = "1m__rsi14__BULL_START_RSI"
BEAR_START_RSI_1m = "1m__rsi14__BEAR_START_RSI"
STO_K_1M = "1m__st14__K"
STO_D_1M = "1m__st14__D"
STO_1M_UP20_ACTIVE_5 = "1m__st14__K_CROSS_UP_20_ACTIVE_5"
STO_1M_DOWN20_ACTIVE_5 = "1m__st14__K_CROSS_DOWN_20_ACTIVE_5"

### 5 Minute Indicators ###
EMA20_5m = "5m__ema__EMA_20"
EMA50_5m = "5m__ema__EMA_50"
EMA75_5m = "5m__ema__EMA_75"
EMA100_5m = "5m__ema__EMA_100"
EMA125_5m = "5m__ema__EMA_125"
EMA150_5m = "5m__ema__EMA_150"
EMA175_5m = "5m__ema__EMA_175"
EMA200_5m = "5m__ema__EMA_200"
MACD_5M = "5m__macd_8_21_5__MACD"
MACD_SIGNAL_5M = "5m__macd_8_21_5__SIGNAL"
MACD_HIST_5M = "5m__macd_8_21_5__HIST"
BULL_DIV_5m = "5m__rsi14__BULL_DIV"
BEAR_DIV_5m = "5m__rsi14__BEAR_DIV"
BULL_START_RSI_5m = "5m__rsi14__BULL_START_RSI"
BEAR_START_RSI_5m = "5m__rsi14__BEAR_START_RSI"
STO_K_5M = "5m__st14__K"
STO_D_5M = "5m__st14__D"
STO_5M_UP20_ACTIVE_3 = "5m__st14__K_CROSS_UP_20_ACTIVE_3"
STO_5M_DOWN20_ACTIVE_3 = "5m__st14__K_CROSS_DOWN_20_ACTIVE_3"


### 15 Minute Indicators ###
EMA20_15m = "15m__ema__EMA_20"
EMA50_15m = "15m__ema__EMA_50"
EMA75_15m = "15m__ema__EMA_75"
EMA100_15m = "15m__ema__EMA_100"
EMA125_15m = "15m__ema__EMA_125"
EMA150_15m = "15m__ema__EMA_150"
EMA175_15m = "15m__ema__EMA_175"
EMA200_15m = "15m__ema__EMA_200"
MACD_15M = "15m__macd_8_21_5__MACD"
MACD_SIGNAL_15M = "15m__macd_8_21_5__SIGNAL"
MACD_HIST_15M = "15m__macd_8_21_5__HIST"
BULL_DIV_15m = "15m__rsi14__BULL_DIV"
BEAR_DIV_15m = "15m__rsi14__BEAR_DIV"
BULL_START_RSI_15m = "15m__rsi14__BULL_START_RSI"
BEAR_START_RSI_15m = "15m__rsi14__BEAR_START_RSI"
STO_K_15M = "15m__st14__K"
STO_D_15M = "15m__st14__D"
STO_15M_UP20_ACTIVE_3 = "15m__st14__K_CROSS_UP_20_ACTIVE_3"
STO_15M_DOWN20_ACTIVE_3 = "15m__st14__K_CROSS_DOWN_20_ACTIVE_3"

### 1 Hour Indicators ###
EMA20_1h = "1h__ema__EMA_20"
EMA50_1h = "1h__ema__EMA_50"
EMA75_1h = "1h__ema__EMA_75"
EMA100_1h = "1h__ema__EMA_100"
EMA125_1h = "1h__ema__EMA_125"
EMA150_1h = "1h__ema__EMA_150"
EMA175_1h = "1h__ema__EMA_175"
EMA200_1h = "1h__ema__EMA_200"
MACD_1H = "1h__macd_8_21_5__MACD"
MACD_SIGNAL_1H = "1h__macd_8_21_5__SIGNAL"
MACD_HIST_1H = "1h__macd_8_21_5__HIST"
BULL_DIV_1h = "1h__rsi14__BULL_DIV"
BEAR_DIV_1h = "1h__rsi14__BEAR_DIV"
BULL_START_RSI_1h = "1h__rsi14__BULL_START_RSI"
BEAR_START_RSI_1h = "1h__rsi14__BEAR_START_RSI"
STO_K_1H = "1h__st14__K"
STO_D_1H = "1h__st14__D"
STO_1H_UP20_ACTIVE_3 = "1h__st14__K_CROSS_UP_20_ACTIVE_3"
STO_1H_DOWN20_ACTIVE_3 = "1h__st14__K_CROSS_DOWN_20_ACTIVE_3"

In [ ]:
# ============================================================
# 1m indicators
# ============================================================

ind_1m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [20, 50, 100, 150, 200, 250],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("stochastic", tag="st14", config=STOCH_CFG),
]


# ============================================================
# 5m indicators
# ============================================================

ind_5m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [20, 50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("stochastic", tag="st14", config=STOCH_CFG),
]


# ============================================================
# 15m indicators
# ============================================================

ind_15m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [20, 50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("stochastic", tag="st14", config=STOCH_CFG),
]

# ============================================================
# 1h indicators
# ============================================================

ind_1h = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [20, 50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("stochastic", tag="st14", config=STOCH_CFG),
]

In [ ]:
# 1m base data
df1 = fetch_binance_klines(
    symbol=symbol,
    interval="1m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df1_feat, _, _ = build_feature_df(
    raw_df=df1,
    tz=tz,
    ma_windows=[],
    indicators=ind_1m,
)


# 5m filter data
df5 = fetch_binance_klines(
    symbol=symbol,
    interval="5m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df5_feat, _, _ = build_feature_df(
    raw_df=df5,
    tz=tz,
    ma_windows=[],
    indicators=ind_5m,
)


# 15m filter data
df15 = fetch_binance_klines(
    symbol=symbol,
    interval="15m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df15_feat, _, _ = build_feature_df(
    raw_df=df15,
    tz=tz,
    ma_windows=[],
    indicators=ind_15m,
)

# 1h filter data
df60 = fetch_binance_klines(
    symbol=symbol,
    interval="1h",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df60_feat, _, _ = build_feature_df(
    raw_df=df60,
    tz=tz,
    ma_windows=[],
    indicators=ind_1h,
)


# Add Stochastic Features
df1_feat = add_stochastic_signal_features(
    df1_feat,
    tag="st14",
    levels=(10, 20, 30),
    windows=(1, 2, 3, 5),
    source="K",
)

df5_feat = add_stochastic_signal_features(
    df5_feat,
    tag="st14",
    levels=(10, 20, 30),
    windows=(1,2, 3, 5),
    source="K",
)

df15_feat = add_stochastic_signal_features(
    df15_feat,
    tag="st14",
    levels=(10, 20, 30),
    windows=(1, 2, 3, 5),
    source="K",
)

df60_feat = add_stochastic_signal_features(
    df60_feat,
    tag="st14",
    levels=(10, 20, 30),
    windows=(1, 2),
    source="K",
)

merged_tmp = align_timeframes(
    base_df=df1_feat,
    other_dfs={
        "5m": df5_feat,
        "15m": df15_feat,
        "1h": df60_feat,
    },
    base_label="1m",
)


In [ ]:


####################################################################################
N_CONFIRM = 1               # number of consecutive confirmation candles required
MAX_WAIT_AFTER_CROSS = 30   # how many candles after the cross we are willing to wait
MACD_THRESHOLD_1M = 100
MACD_THRESHOLD_5M = 10
MACD_THRESHOLD_15M = 10 

MIN_MACD_DIFF_1M = 5
MIN_MACD_DIFF_5M = 5
MIN_MACD_DIFF_15M = 5

DIV_LOOKBACK = 5

N_SPREAD_CONFIRM = 3
MIN_SPREAD_PCT = 0.1

MIN_EMAS_CROSSED = 4
CROSS_LOOKBACK = 40


#####################################################################################


LONG_EMA_FILTERS = [
    EMA20_1m
    # EMA50_1m,
    # EMA100_1m,
    # EMA150_1m,
    # EMA200_1m,
    # EMA100_5m,
    # EMA200_5m,
    # EMA100_15m,
    #EMA200_15m
]


macd_rules = ALL(
    Rule(
        "1m MACD above threshold",
        lambda c: c.gt(MACD_1M, 10)
    ),
    
    # Rule(
    #     "5m MACD above threshold",
    #     lambda c: c.gt(MACD_5M, MACD_THRESHOLD_5M)
    # ),
    
    # Rule(
    #     "15m MACD above threshold",
    #     lambda c: c.gt(MACD_15M, MACD_THRESHOLD_15M)
    # ),

    Rule(
        "1m MACD - Signal above minimum",
        lambda c: c.gt(MACD_HIST_1M, 50)
    ),
    
    Rule(
         "5m MACD - Signal above minimum",
         lambda c: c.gt(MACD_HIST_5M, 20)
     ),

    Rule(
        "15m MACD - Signal above minimum",
        lambda c: c.gt(MACD_HIST_15M, 20)
    ),
    
)


open_rules_long = ALL(
    macd_rules,
    Rule(
        "Current close above all EMA filters",
        lambda c: c.close_below_all(LONG_EMA_FILTERS)

),
    Rule(
        f"Last {N_CONFIRM} candles green",
        lambda c: c.consecutive_green(n=N_CONFIRM)
    ),
)



close_rules_long = ALL(
    Rule(
        "Current close above all EMA filters",
        lambda c: c.close_below_all(LONG_EMA_FILTERS)

),
)




###################### stochastic  rules #########################


X_1M = 5
Y_5M = 6
Z_15M = 3

open_rules_long_stoch_recovery = ALL(

    Rule(
        "1m MACD above threshold",
        lambda c: c.gt(MACD_1M, 100)
    ),
    
    # Rule(
    #     "5m MACD above threshold",
    #     lambda c: c.gt(MACD_5M, 5)
    # ),
    
    Rule(
        "Current close above all EMA filters",
        lambda c: c.close_above_all(LONG_EMA_FILTERS)

    ),
    # Rule(
    #     "1m MACD - Signal above minimum",
    #     lambda c: c.lt(EMA50_5m, EMA100_5m)
    # ),
    # Rule(
    #     f"1m stochastic K crossed up 20 in last {1} 1m candles and stayed above",
    #     lambda c: c.flag(f"1m__st14__K_CROSS_UP_20_ACTIVE_{1}")
    # ),

    Rule(
        f"5m stochastic K crossed up 20 in last {Y_5M} 5m candles and stayed above",
        lambda c: c.flag(f"5m__st14__K_CROSS_UP_20_ACTIVE_{Y_5M}")
    ),

    Rule(
        "15m Stochastic K > D",
        lambda c: c.gt(STO_K_15M, STO_D_15M)
    ),
    Rule(
        "5m Stochastic K > D",
        lambda c: c.gt(STO_K_5M, STO_D_5M)
    ),
    # Rule(
    #     f"15m stochastic K crossed up 20 in last {Z_15M} 15m candles and stayed above",
    #     lambda c: c.flag(f"15m__st14__K_CROSS_UP_20_ACTIVE_{Z_15M}")
    # ),
    Rule(
        f"1m stochastic K crossed up 20 in last {1} 1m candles and stayed above",
        lambda c: c.flag(f"1h__st14__K_CROSS_UP_20_ACTIVE_{1}")
    )
)

close_rules_long_stoch_recovery = ANY(
    Rule(
        "1m stochastic K crossed back below 20",
        lambda c: c.flag("1m__st14__K_CROSS_DOWN_20")
    ),

    Rule(
        "1m stochastic K crossed down from overbought 80",
        lambda c: c.flag("1m__st14__K_CROSS_DOWN_80")
    ),
)




In [ ]:
# ============================================================
# EMA compression feature names
# ============================================================

EMA_1M_COMP_NAME = "ema_1m_50_100_150_comp"
EMA_5M_COMP_NAME = "ema_5m_50_100_150_comp"
EMA_15M_COMP_NAME = "ema_15m_50_100_150_comp"
EMA_1H_COMP_NAME = "ema_1h_50_100_150_comp"

EMA_1M_RANGE = f"{EMA_1M_COMP_NAME}__range_pct"
EMA_5M_RANGE = f"{EMA_5M_COMP_NAME}__range_pct"
EMA_15M_RANGE = f"{EMA_15M_COMP_NAME}__range_pct"

N_1M_SLOPE = 3
N_5M_SLOPE = 2
N_15M_SLOPE = 2

# ============================================================
# Add EMA compression features for 5m, 15m, 1h
# ============================================================

merged = add_ema_compression_features(
    merged_tmp,
    specs=[
        EMACompressionSpec(
            name=EMA_1M_COMP_NAME,
            cols=[EMA50_1m, EMA100_1m, EMA150_1m],
            compress_thr=0.35,
            expand_thr=0.40,
            lookback=20,
            require_bullish_order=False,
            fresh_only=False,
        ),
        EMACompressionSpec(
            name=EMA_5M_COMP_NAME,
            cols=[EMA50_5m, EMA100_5m, EMA150_5m],
            compress_thr=0.15, #0.15
            expand_thr=0.3, #0.3
            lookback=20, #20
            require_bullish_order=True,
            fresh_only=False,
        ),

        EMACompressionSpec(
            name=EMA_15M_COMP_NAME,
            cols=[EMA50_15m, EMA100_15m, EMA150_15m],
            compress_thr=0.10,
            expand_thr=0.15,
            lookback=20,
            require_bullish_order=True,
            fresh_only=False,
        ),

        EMACompressionSpec(
            name=EMA_1H_COMP_NAME,
            cols=[EMA50_1h, EMA100_1h, EMA150_1h],
            compress_thr=0.10,
            expand_thr=0.15,
            lookback=20,
            require_bullish_order=True,
            fresh_only=False,
        ),
    ],
)

######## ema rules #####
LONG_EMA_FILTERS = [
    EMA20_1m,
    EMA50_15m
]

COMPRESSION_MAX = 0.7
COMPRESSION_MIN = 0.35

open_rules_long = ALL(
    Rule(
        "Current close above all EMA filters",
        lambda c: c.close_above_all([EMA100_5m])

    ),
    Rule(
        "1m MACD - Signal above minimum",
        lambda c: c.gt(EMA100_5m, EMA200_15m)
    ),
    Rule(
        "15m EMA50/100/150 compressed then expanded bullish",
        lambda c: c.flag(f"{EMA_1M_COMP_NAME}__bull_breakout")
    ),
    Rule(
        f"15m EMA compression between {COMPRESSION_MIN}% and {COMPRESSION_MAX}%",
        lambda c: (
            c.gte(f"{EMA_5M_COMP_NAME}__range_pct", COMPRESSION_MIN)
            and c.lte(f"{EMA_5M_COMP_NAME}__range_pct", COMPRESSION_MAX)
        )
    ),
)

# open_rules_long = ALL(
#     Rule(
#         "15m EMA50/100/150 compressed then expanded bullish",
#         lambda c: c.flag(f"{EMA_5M_COMP_NAME}__bull_breakout")
#     ),
# )


### slope rules
open_slope_rules_long = ALL(
    Rule(
        "1m EMA range % strongly rising",
        lambda c: range_slope_strong_for_n_tf_candles(
            c, EMA_1M_RANGE, tf_step=1, n=3,
            min_step_delta=0.003,
            min_total_delta=0.010,
        )
    ),

    Rule(
        "5m EMA range % strongly rising",
        lambda c: range_slope_strong_for_n_tf_candles(
            c, EMA_5M_RANGE, tf_step=5, n=2,
            min_step_delta=0.005,
            min_total_delta=0.015,
        )
    ),

    Rule(
        "15m EMA range % strongly rising",
        lambda c: range_slope_strong_for_n_tf_candles(
            c, EMA_15M_RANGE, tf_step=15, n=2,
            min_step_delta=0.005,
            min_total_delta=0.015,
        )
    ),


    Rule(
        "1m EMA ribbon bullish",
        lambda c: c.gt(EMA50_1m, EMA100_1m) and c.gt(EMA100_1m, EMA150_1m)
    ),

    Rule(
        "5m EMA ribbon bullish",
        lambda c: c.gt(EMA50_5m, EMA100_5m) and c.gt(EMA100_5m, EMA150_5m)
    ),

    Rule(
        "15m EMA ribbon bullish",
        lambda c: c.gt(EMA50_15m, EMA100_15m) and c.gt(EMA100_15m, EMA150_15m)
    ),
)

close_rules_long = ALL(
    Rule(
        "Close below 1m EMA50",
        lambda c: c.lt("close", EMA50_5m)
    ),
)
##################################################

strategy = Strategy(
    open_rules_long=open_slope_rules_long,
    close_rules_long=close_rules_long,
)

cfg = SimConfig(
    initial_cash=10_000,
    max_open_trades=1,
    fee_bps=10,
    slippage_bps=1,

    sim_start="2025-11-01",
    sim_end="2026-05-25",
    sim_tz=tz,

    exit=TradeExitConfig(
        enabled=True,
        # Important:
        # if both TP and SL are touched in same candle, assume SL first
        intrabar_priority="stop_first",
        # Important:
        # disables ST2 close rule while TP/SL system is active
        allow_rule_close=False,

        stop_loss=StopLossConfig(
            mode="entry_pct",
            value=1.25,   # 0.5% below entry for long
        ),
        sizing=PositionSizingConfig(
            mode="cash",  # use normal full cash sizing
        ),
        take_profits=(
            TakeProfitConfig(
                label="TP1",
                mode="entry_pct",
                value=1.5,          # +0.5%
                close_pct=100.0,     # close 50% of remaining
                # move_stop_mode="breakeven",
            ),

        ),
    ),
)

res = run_simulation(
    df=merged,
    strategy=strategy,
    cfg=cfg,
    time_col="t",
    price_col="close",
)

res.stats #, res.events.head(), res.equity_curve.tail()

In [ ]:
trade_summary_table(res.trades, initial_cash=10_000, interval="1m")

In [ ]:
ema_compression_specs = {
    "1m": IndicatorSpec("ema_compression", tag=EMA_1M_COMP_NAME, config={
        "feature_name": EMA_1M_COMP_NAME,
        "title": "1m EMA50/100/150 Compression (%)",
    }),

    "5m": IndicatorSpec("ema_compression", tag=EMA_5M_COMP_NAME, config={
        "feature_name": EMA_5M_COMP_NAME,
        "title": "5m EMA50/100/150 Compression (%)",
    }),

    "15m": IndicatorSpec("ema_compression", tag=EMA_15M_COMP_NAME, config={
        "feature_name": EMA_15M_COMP_NAME,
        "title": "15m EMA50/100/150 Compression (%)",
    }),

    "1h": IndicatorSpec("ema_compression", tag=EMA_1H_COMP_NAME, config={
        "feature_name": EMA_1H_COMP_NAME,
        "title": "1h EMA50/100/150 Compression (%)",
    }),
}

In [ ]:
indicators_by_tf = {
    "1m": [*ind_1m, ema_compression_specs["1m"]],
    "5m": [*ind_5m, ema_compression_specs["5m"]],
    "15m": [*ind_15m, ema_compression_specs["15m"]],
    "1h": [*ind_1h, ema_compression_specs["1h"]],
}

plot_toggles = [
    # Optional EMA overlays
    PlotToggle("1m", "ema", visible="legendonly"),
    PlotToggle("5m", "ema", visible="legendonly"),
    PlotToggle("15m", "ema", visible="legendonly"),
    PlotToggle("1h", "ema", visible="legendonly"),

    # Stochastic panels
    PlotToggle("1m", "st14", visible=False),
    PlotToggle("5m", "st14", visible=False),
    PlotToggle("15m", "st14", visible=False),
    PlotToggle("1h", "st14", visible=False),

    # Hide MACD
    PlotToggle("1m", "macd_8_21_5",  visible=False),
    PlotToggle("5m", "macd_8_21_5",  visible=False),
    PlotToggle("15m", "macd_8_21_5",  visible=False),
    PlotToggle("1h", "macd_8_21_5",  visible=False),

    PlotToggle("1m", EMA_1M_COMP_NAME, visible=True),
    PlotToggle("5m", EMA_5M_COMP_NAME, visible=True),
    PlotToggle("15m", EMA_15M_COMP_NAME, visible=True),
    PlotToggle("1h", EMA_1H_COMP_NAME, visible=True),

]

plot_indicators = make_plot_indicators_from_toggles(
    indicators_by_tf=indicators_by_tf,
    toggles=plot_toggles,
)

assert_plot_columns_exist(merged, plot_indicators)


In [ ]:
plot_simulation(
    df_raw=merged,
    symbol=symbol,
    interval="1m",
    market=market,
    trades=res.trades,
    ma_windows=[],
    indicators=plot_indicators,
    plot_cfg=PlotConfig(
        tz=tz,
        # date_from="2026-03-23",
        # date_to="2026-03-24",
        date_from="2025-11-9",
        date_to="2025-11-12",
        height=1400,
        width=1900,
    ),
)
